# Preprocessing Pipeline
Transform raw ATIS data into clean, model-ready files for the rest of the pipeline.    

## What this notebook does:
*   **Loads and inspects** raw `atis.json`
*   **Flattens** nested JSON into question-SQL pairs using sentence-level variable substitution
*   Takes `sql[0]` as the **"gold SQL"** for each entry
*   **Deduplicates** on (question, SQL) pair and strips trailing semicolons from SQL
*   Saves `split_train.jsonl`, `split_dev.jsonl`, `split_test.jsonl` to `data/`
*   Schema files were built manually (see `schemas/` in the GitHub Repo)

## To run this notebook:
*   Simply click "Run all"

## Outputs:
*   `split_train.jsonl`, `split_dev.jsonl`, `split_test.jsonl`are saved under `data/` folder of the Google Colab session files.
---

The ATIS data came from this repository: https://github.com/jkkummerfeld/text2sql-data.git  

---

**Final dataset:** 4895 unique question-SQL pairs (4055 train / 436 dev / 404 test)  


In [1]:
import os
import json
from pathlib import Path
import urllib.request
from collections import Counter
import sys

# Data path
DATA_DIR = Path("data")

# Create data output folder if they don't exist
DATA_DIR.mkdir(exist_ok=True)

print("Project folders created/exist.")
print(f"Working directory: {os.getcwd()}")

Project folders created/exist.
Working directory: /content


## 1) Load and Inspect "Raw" Data

> If GitHub download for `schema_relational.txt` fails in the next cell, manually upload to you Colab session (Files icon -> Upload) and re-run. The schema file should be in the zip file under `schemas/`. Don't upload the schema folder, just the files inside it.

In [2]:
# Raw dataset GitHub file
ATIS_URL = "https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/refs/heads/master/data/atis.json"
ATIS_PATH = DATA_DIR / "atis.json"

if not ATIS_PATH.exists():
  print ("Downloading atis.json...")
  urllib.request.urlretrieve(ATIS_URL, ATIS_PATH)
  print("Download complete.")
else:
  print("atis.json already exists, skipping download.")

print(f"Raw data path: {ATIS_PATH}")

# Download schema_relational.txt from my GitHub repo
SCHEMA_URL = "https://raw.githubusercontent.com/halleepham/text2sql-schema-representation/refs/heads/main/schemas/schema_relational.txt"
SCHEMA_PATH = Path("schema_relational.txt")

if not SCHEMA_PATH.exists():
  print("\nDownloading schema_relational.txt...")
  urllib.request.urlretrieve(SCHEMA_URL, SCHEMA_PATH)
  print("Download complete.")
else:
  print("schema_relational.txt already exists, skipping download.")

Download complete.
Raw data path: data/atis.json

Download complete.


In [3]:
# Read raw json file into python dictionary
with open(ATIS_PATH, "r") as f:
  raw_data = json.load(f)

print(f"Total entries: {len(raw_data)}")
print(f"\nKeys in each entry: {list(raw_data[0].keys())}")
print(f"\nKeys in each sentence: {list(raw_data[0]['sentences'][0].keys())}")

Total entries: 947

Keys in each entry: ['comments', 'old-name', 'query-split', 'sentences', 'sql', 'variables']

Keys in each sentence: ['text', 'question-split', 'variables']


In [4]:
print(f"\nExample entry:")
print(json.dumps(raw_data[0], indent=2))


Example entry:
{
  "comments": [],
  "old-name": "",
  "query-split": "train",
  "sentences": [
    {
      "text": "list all the flights that arrive at airport_code0 from various cities",
      "question-split": "train",
      "variables": {
        "airport_code0": "MKE"
      }
    },
    {
      "text": "what flights from any city land at airport_code0",
      "question-split": "train",
      "variables": {
        "airport_code0": "MKE"
      }
    },
    {
      "text": "show me the flights into airport_code0",
      "question-split": "train",
      "variables": {
        "airport_code0": "DAL"
      }
    },
    {
      "text": "show me the flights arriving at airport_code0",
      "question-split": "train",
      "variables": {
        "airport_code0": "DAL"
      }
    },
    {
      "text": "list all the flights that arrive at airport_code0",
      "question-split": "train",
      "variables": {
        "airport_code0": "MKE"
      }
    },
    {
      "text": "list all the 

Output Notes:

*   **947 total entries** (query groups)
*   Each sentence may have **multiple phrasings**. The first entry has has 21 phrasings.
*   `sql` is a list so there are **multiple SQL variants**. This entry has 3 different variants
*   Each sentence has its own `question_split` label (`train`/`dev`/`test`) and its own `variables` dictionary for substitution
*   Variables placeholders appear in both question text (unquoted) and SQL (quoted). Substitution must handle both.



## 2) Flatten and Substitute Variables
For each entry:  
*   `sql[0]` is taken as the gold SQL
*   For each sentence phrasing, sentence-level variables values are subsituted into both the question text and SQL
*   Rows are collected as `{question, sql, split}`


In [5]:
def substitute_variables(text, variables):
  """Takes the text (question) and subsitutes with the variables"""
  # Loop over each key-value pair in variables dictionary
  for placeholder, value in variables.items():
    text = text.replace(f'"{placeholder}"', f'"{value}"') # Replace quoted placeholder first (SQL) to avoid replacement mistakes
    text = text.replace(placeholder, value) # then unquoted (question text)
  return text

rows = []
for entry in raw_data:
  sql_template = entry["sql"][0]  # take first SQL variant only

  # Loop overy every sentence phrasing
  for sentence in entry["sentences"]:
    variables = sentence["variables"]
    question = substitute_variables(sentence["text"], variables)
    sql = substitute_variables(sql_template, variables)
    split = sentence["question-split"]
    rows.append({"question": question, "sql": sql, "split": split})

print(f"Total rows before cleaning: {len(rows)}")
print(f"\nExample row:")
print(json.dumps(rows[0], indent=2))

Total rows before cleaning: 5280

Example row:
{
  "question": "list all the flights that arrive at MKE from various cities",
  "sql": "SELECT DISTINCT FLIGHTalias0.FLIGHT_ID FROM AIRPORT AS AIRPORTalias0 , AIRPORT_SERVICE AS AIRPORT_SERVICEalias0 , CITY AS CITYalias0 , FLIGHT AS FLIGHTalias0 WHERE AIRPORTalias0.AIRPORT_CODE = \"MKE\" AND CITYalias0.CITY_CODE = AIRPORT_SERVICEalias0.CITY_CODE AND FLIGHTalias0.FROM_AIRPORT = AIRPORT_SERVICEalias0.AIRPORT_CODE AND FLIGHTalias0.TO_AIRPORT = AIRPORTalias0.AIRPORT_CODE ;",
  "split": "train"
}


Output notes:
*   `sql` still has trailing semicolon, will be taken care of
*   `sql` still has extra spaces around commas and operators.
*   `AIRPORT AS AIRPORTalias0 , ` is the original canonicalization from dataset, will leave as is


## 3) Clean and Filter
*   Strip trailing semicolons and whitespace from `sql`
*   Deduplicate on (question, SQL) pair

**Note:** I stripped the semi colons so target outputs are consistent across all examples. For exact match evaluation it could cause false negatives if the model sometimes generates a semicolon and sometimes doesn't. Its a minor issue since dataset is already canonicalized, but stripping is cleaner and safer.

In [10]:
multi_statement = [r for r in rows if r["sql"].strip().rstrip(";").strip().count(";") > 0]
print(f"Queries with internal semicolons: {len(multi_statement)}")

Queries with internal semicolons: 0


In [6]:
seen = set()
cleaned = []

for row in rows:
  # Strip trailing whitespace and semicolon from SQL
  sql = row["sql"].rstrip(";").strip()
  question = row["question"].strip()

  # Deduplicate on (question, SQL) pair
  key = (question, sql)
  if key in seen:
    continue
  seen.add(key)
  cleaned.append({"question": question, "sql": sql, "split": row["split"]})

print(f"Rows before cleaning: {len(rows)}")
print(f"Rows after cleaning: {len(cleaned)}")
print(f"\n{len(rows) - len(cleaned)} rows duplicates removed.")

Rows before cleaning: 5280
Rows after cleaning: 4895

385 rows duplicates removed.


Output Notes:
*   **385 duplicate (question, SQL) pairs removed**. These were exact duplicates produced during variable subtitution where different sentence phrasings with different variable values became the same.
*   **4895 unique question-SQL pairs** across train, dev, and test splits

## 4) Save Splits
Separates cleaned rows by existing `split` label and write to JSONL files in `data/`. `dev` is another name for "val" (validation set)  

---

*   The original `question-split` labels from the dataset are used rather than random reshuffling. The dataset authors designed these splits to prevent leakeage (similar phrasing of the same underlying query are intentionally distributed across splits).  
*   Random splitting would break this and could place near-duplicate questions in both train and test, which may lead to artifically inflating evaluation metrics.

In [7]:
# Count rows per split
split_counts = Counter([row["split"] for row in cleaned])
print(f"Split counts: {dict(split_counts)}")

# Separate rows by split
splits = {"train": [], "dev": [], "test": []}
for row in cleaned:
  splits[row["split"]].append(row)

# Write each split to a JSONL file
for split_name, split_rows in splits.items():
  out_path = DATA_DIR / f"split_{split_name}.jsonl"
  with open(out_path, "w") as f:
    for row in split_rows:
      f.write(json.dumps(row) + "\n")   # one json object per line
  print(f"Saved {len(split_rows)} rows to {out_path.name}")

print(f"\nTrain percentage: {(len(splits["train"]) / len(cleaned)) * 100:.2f}%")
print(f"Dev percentage: {(len(splits["dev"]) / len(cleaned)) * 100:.2f}%")
print(f"Test percentage: {(len(splits["test"]) / len(cleaned)) * 100:.2f}%")

Split counts: {'train': 4055, 'dev': 436, 'test': 404}
Saved 4055 rows to split_train.jsonl
Saved 436 rows to split_dev.jsonl
Saved 404 rows to split_test.jsonl

Train percentage: 82.84%
Dev percentage: 8.91%
Test percentage: 8.25%


Output Notes:


*   Split distribution is heavily weighted towards `train` (typical for NLP datasets)
*   This split is different from my original proposal (~400 dev and ~600 test). But I will keep as is because the original split is designed to avoid "leakage" between similar phrasings.



## 5) Schema Files
All four schema representation files were built manually using `atis-db.sql` as the column reference. Files were verified by Claude after creation to find spelling errors or and missing columns.
*   `schema_relational.txt`: relational format, table and column names only
*   `schema_create_table.txt`: CREATE TABLE statements with data types and primary keys
*   `schema_json.txt`: one JSON object per table with column list
*   `schema_nl.txt`: natural language description of each table and its columns

NOTE: no foreign key relationships are included in any format. The source database has no formally defined foreign key constraints.  
You can see all schema files in my GitHub repository here: https://github.com/halleepham/text2sql-schema-representation/tree/main/schemas

## 6) Verification
Verifies saved splits and schema load correctly and confirm prompt assembly works end to end.

In [8]:
# Prompt Builder Functions

def load_schema(schema_path):
  """Load schema string from a .txt file."""
  with open(schema_path, "r") as f:
    return f.read().strip()

def build_prompt(question, schema):
  """Assemble a prompt from a question and schema string."""
  return(
      f"Translate the following question into SQL.\n\n"
      f"Schema:\n{schema}\n\n"
      f"Question: {question}\n\n"
      f"SQL:"
  )

In [9]:
# Load schema
schema = load_schema(Path("schema_relational.txt"))

# Load a few rows from dev split
dev_rows = []
with open(DATA_DIR / "split_dev.jsonl", "r") as f:
  for i, line in enumerate(f):
    if i >= 3:  # load first 3 rows only
      break
    dev_rows.append(json.loads(line))

# Print example prompts
for i, row in enumerate(dev_rows):
  prompt = build_prompt(row["question"], schema)
  print(f"\n-------Example {i+1}-------")
  print(prompt)
  print(f"\nGold SQL: {row["sql"]}")


-------Example 1-------
Translate the following question into SQL.

Schema:
aircraft(aircraft_code, aircraft_description, manufacturer, basic_type, engines, propulsion, wide_body, wing_span, length, weight, capacity, pay_load, cruising_speed, range_miles, pressurized)
airline(airline_code, airline_name, note)
airport(airport_code, airport_name, airport_location, state_code, country_name, time_zone_code, minimum_connect_time)
airport_service(city_code, airport_code, miles_distant, direction, minutes_distant)
city(city_code, city_name, state_code, country_name, time_zone_code)
class_of_service(booking_class, rank, class_description)
code_description(code, description)
compartment_class(compartment, class_type)
date_day(month_number, day_number, year, day_name)
days(days_code, day_name)
dual_carrier(main_airline, low_flight_number, high_flight_number, dual_airline, service_name)
equipment_sequence(aircraft_code_sequence, aircraft_code)
fare(fare_id, from_airport, to_airport, fare_basis_c

Output notes:
*   Schema loads cleanly and formats correctly in prompts
*   Questions have real values substituted in (MKE, DAL)
*   The three examples are all slightly different phrasings of the same underlying query, so dataset diversity is preserved and deduplication worked correctly